In [1]:
import pandas as pd

# Load files
diagnoses = pd.read_csv(r"diagnoses_icd.csv")
icd_lookup = pd.read_csv(r"C:\Users\Warner_Beast\OneDrive\Documents\CUNY\DATA 698\Data\Manual\icd-9-comparison-excel-file.csv")

# Inspect columns
print("Diagnoses columns:", diagnoses.columns.tolist())
print("ICD lookup columns:", icd_lookup.columns.tolist())

Diagnoses columns: ['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']
ICD lookup columns: ['ICD9', 'Description', 'HCUP CCS Classification', 'HCC Category', 'CCW Classification', 'CDPS Classification']


This code prepares and enriches the diagnosis dataset by standardizing ICD codes and linking them to their corresponding descriptions. First, it filters the diagnosis table to keep only records with ICD-9 codes, ensuring consistency with the ICD-9 reference file. Next, a function called normalize_icd() is created to clean and standardize the diagnosis codes by removing dots, trimming spaces, and stripping leading zeros so that codes from different datasets can match correctly. The normalized codes are then added to both the diagnosis dataset and the ICD lookup table. After standardization, the two datasets are merged using the normalized ICD code, which attaches diagnosis descriptions and category information to each diagnosis record. Finally, the code calculates a match rate to verify how many diagnoses successfully linked to a description and displays the first ten rows of the merged dataset for inspection.

In [2]:

# ---- 1) Keep only ICD-9 rows from diagnoses ----
diagnoses = diagnoses[diagnoses["icd_version"] == 9].copy()

# ---- 2) Normalize ICD codes (remove dots + remove leading zeros for safer matching) ----
def normalize_icd(x):
    x = str(x).replace(".", "").strip()
    x = x.lstrip("0")  # helps match codes like "07070" to "7070" if lookup stores without leading zeros
    return x

diagnoses["icd9_norm"] = diagnoses["icd_code"].apply(normalize_icd)
icd_lookup["icd9_norm"] = icd_lookup["ICD9"].apply(normalize_icd)

# ---- 3) Merge to bring in descriptions/categories ----
dx = diagnoses.merge(
    icd_lookup,
    on="icd9_norm",
    how="left"
)

# Quick check of match rate
match_rate = dx["Description"].notna().mean()
print(f"Match rate: {match_rate:.1%}")

dx.head(10)
#

Match rate: 99.6%


,subject_id,hadm_id,seq_num,icd_code,icd_version,icd9_norm,ICD9,Description,HCUP CCS Classification,HCC Category,CCW Classification,CDPS Classification
0,10000032,22595853,1,5723,9,5723,5723,Portal hypertension,Oth liver dx,HCC25 End-Stage Liver Disease,NaN,"Gastrointestinal, Medium"
1,10000032,22595853,2,78959,9,78959,78959,Ascites NEC,Oth liver dx,NaN,NaN,"Gastrointestinal, Medium"
2,10000032,22595853,3,5715,9,5715,5715,Cirrhosis of liver NOS,Oth liver dx,HCC26 Cirrhosis of Liver,NaN,"Gastrointestinal, Medium"
3,10000032,22595853,4,07070,9,7070,7070,Hpt C w/o hepat coma NOS,Hepatitis,NaN,NaN,"Infectious Disease, Low"
4,10000032,22595853,5,496,9,496,496,Chr airway obstruct NEC,COPD,HCC108 Chronic Obstructive Pulmonary Disease,COPD,"Pulmonary, Low"
5,10000032,22595853,6,29680,9,29680,29680,Bipolar disorder NOS,Mood disorders,"HCC55 Major Depressive, Bipolar, and Paranoid ...",NaN,"Psychiatric, Medium Low"
6,10000032,22595853,7,30981,9,30981,30981,Posttraumatic stress dis,Anxiety disord,NaN,NaN,"Psychiatric, Medium Low"
7,10000032,22595853,8,V1582,9,V1582,V1582,History of tobacco use,Screening and,NaN,NaN,NaN
8,10000032,22841357,1,07071,9,7071,7071,Hpt C w hepatic coma NOS,Hepatitis,NaN,NaN,"Infectious Disease, Low"
9,10000032,22841357,2,78959,9,78959,78959,Ascites NEC,Oth liver dx,NaN,NaN,"Gastrointestinal, Medium"


In [3]:
unmatched = dx[dx["Description"].isna()][["icd_code", "icd9_norm"]].drop_duplicates().head(30)
unmatched

#

,icd_code,icd9_norm
2741,V854,V854
2877,2766,2766
3232,1737,1737
3431,0414,414
3571,9974,9974
4898,9690,9690
7427,5185,5185
7550,4440,4440
7744,5968,5968
7964,5997,5997


In [4]:
# Diagnosis count per admission
dx_count = dx.groupby("hadm_id")["icd9_norm"].nunique().reset_index(name="dx_count")

# 3-digit grouping (rough category feature)
dx["dx3"] = dx["icd9_norm"].str[:3]
top_dx3 = dx["dx3"].value_counts().head(20).index

# Flag whether admission had any of the top 20 groups
dx_top = (
    dx.assign(is_top=dx["dx3"].isin(top_dx3).astype(int))
      .pivot_table(index="hadm_id", columns="dx3", values="is_top", aggfunc="max", fill_value=0)
      .reset_index()
)

dx_count.head(), dx_top.head()

(    hadm_id  dx_count
 0  20000019        12
 1  20000041        10
 2  20000057        23
 3  20000102         3
 4  20000235        18,
 dx3   hadm_id  100  104  108  110  111  112  114  115  116  ...  V82  V83  \
 0    20000019    0    0    0    0    0    0    0    0    0  ...    0    0   
 1    20000041    0    0    0    0    0    0    0    0    0  ...    0    0   
 2    20000057    0    0    0    0    0    0    0    0    0  ...    0    0   
 3    20000102    0    0    0    0    0    0    0    0    0  ...    0    0   
 4    20000235    0    0    0    0    0    0    0    0    0  ...    0    0   
 
 dx3  V84  V85  V86  V87  V88  V89  V90  V91  
 0      0    0    0    0    0    0    0    0  
 1      0    0    0    0    0    0    0    0  
 2      0    0    0    0    0    0    0    0  
 3      0    0    0    0    0    0    0    0  
 4      0    0    0    0    0    0    0    0  
 
 [5 rows x 968 columns])

In [5]:
distinct_categories = dx["Description"].dropna().unique()
distinct_categories

array(['Portal hypertension', 'Ascites NEC', 'Cirrhosis of liver NOS',
       ..., 'Benign neo pharynx NOS', 'Cor triatriatum',
       'Nonadmin necess medicine'], dtype=object)

In [6]:
distinct_categories = dx["Description"].dropna().unique()

for item in sorted(distinct_categories):
    print(item)

"ventilation" pneumonit
1 deg burn back of hand
1 eye-sev/oth-blind NOS
10-19% bdy brn/10-19% 3d
10-19% bdy brn/3 deg NOS
1st deg burn abdomn wall
1st deg burn back
1st deg burn chest wall
1st deg burn ear
1st deg burn finger
1st deg burn forearm
1st deg burn genitalia
1st deg burn hand NOS
1st deg burn head NOS
1st deg burn leg NOS
1st deg burn leg-mult
1st deg burn lower leg
1st deg burn neck
1st deg burn scalp
1st deg burn thigh
2 deg burn back of hand
20-29% bdy brn/3 deg NOS
25-26 comp wks gestation
2nd deg burn abdomn wall
2nd deg burn ankle
2nd deg burn arm NOS
2nd deg burn back
2nd deg burn breast
2nd deg burn face NEC
2nd deg burn finger
2nd deg burn foot
2nd deg burn forearm
2nd deg burn hand NOS
2nd deg burn hand-mult
2nd deg burn head NOS
2nd deg burn lower leg
2nd deg burn mult finger
2nd deg burn mult site
2nd deg burn palm
2nd deg burn scalp
2nd deg burn shoulder
2nd deg burn thigh
2nd deg burn thumb
2nd deg burn trunk NEC
3rd deg burn abdomn wall
3rd deg burn arm-mult
3

In [7]:
distinct_df = pd.DataFrame(sorted(dx["Description"].dropna().unique()), columns=["Diagnosis_Description"])
distinct_df

,Diagnosis_Description
0,"""ventilation"" pneumonit"
1,1 deg burn back of hand
2,1 eye-sev/oth-blind NOS
3,10-19% bdy brn/10-19% 3d
4,10-19% bdy brn/3 deg NOS
...,...
9113,Xeroderma of eyelid
9114,Yatapoxvirus infectn NOS
9115,Yoga
9116,Zoonotic bact dis NOS


In [10]:
dx["Description"].nunique()

9118

In [9]:
dx.to_csv("diagnosis_main.csv", index=False)

In [11]:
# Normalize first (if not already done)
dx["icd9_norm"] = dx["icd_code"].astype(str).str.replace(".", "", regex=False)

# Hypertension = codes starting with 401–405
dx["hypertension_flag"] = dx["icd9_norm"].str.startswith(
    ("401", "402", "403", "404", "405")
).astype(int)

In [12]:
htn_patients = dx.loc[dx["hypertension_flag"] == 1, "subject_id"].nunique()
htn_patients

58946

In [13]:
htn_admission = (
    dx.groupby("hadm_id")["hypertension_flag"]
      .max()
      .reset_index()
)

In [14]:
admissions = pd.read_csv(r"admissions.csv")

In [15]:
#admissions = pd.read_csv("/mnt/data/admissions.csv")

admissions = admissions.merge(
    htn_admission,
    on="hadm_id",
    how="left"
)

# Fill missing as 0 (no hypertension)
admissions["hypertension_flag"] = admissions["hypertension_flag"].fillna(0).astype(int)

In [16]:
admissions["hypertension_flag"].value_counts()

0    296468
1    134620
Name: hypertension_flag, dtype: int64

In [18]:
# Ensure normalized code exists
dx["icd9_norm"] = dx["icd_code"].astype(str).str.replace(".", "", regex=False)

# Extract first 3 digits
dx["dx3"] = dx["icd9_norm"].str[:3]

# Convert to numeric where possible
dx["dx3_num"] = pd.to_numeric(dx["dx3"], errors="coerce")

# Cardio range 390–459
cardio_dx = dx[(dx["dx3_num"] >= 390) & (dx["dx3_num"] <= 459)]

# Unique cardiovascular diagnoses
sorted(cardio_dx["Description"].dropna().unique())

['AMI NEC, initial',
 'AMI NEC, subsequent',
 'AMI NOS, initial',
 'AMI NOS, subsequent',
 'AMI NOS, unspecified',
 'AMI anterior wall, init',
 'AMI anterior wall,subseq',
 'AMI anterior wall,unspec',
 'AMI anterolateral, init',
 'AMI anterolateral,subseq',
 'AMI inferior wall, init',
 'AMI inferior wall,subseq',
 'AMI inferior wall,unspec',
 'AMI inferolateral, init',
 'AMI inferolateral,subseq',
 'AMI inferopost, initial',
 'AMI inferopost, subseq',
 'AMI lateral NEC, initial',
 'AMI lateral NEC, subseq',
 'Abdom aortic aneurysm',
 'Abdominal aortic ectasia',
 'Ac DVT/emb distl low ext',
 'Ac DVT/emb prox low ext',
 'Ac DVT/embl low ext NOS',
 'Ac DVT/embl up ext',
 'Ac cerebrovasc insuf NOS',
 'Ac diastolic hrt failure',
 'Ac embl internl jug vein',
 'Ac embl subclav veins',
 'Ac embl suprfcl up ext',
 'Ac embl thorac vein NEC',
 'Ac emblsm axillary veins',
 'Ac emblsm up ext NOS',
 'Ac embolism veins NEC',
 'Ac idiopath pericarditis',
 'Ac ischemic hrt dis NEC',
 'Ac myocardit in o

In [19]:
cardio_patients = cardio_dx["subject_id"].nunique()
cardio_patients

75211

In [20]:
dx["heart_failure"] = dx["icd9_norm"].str.startswith("428").astype(int)

In [21]:
dx["stroke"] = dx["dx3_num"].between(430, 438).astype(int)

In [22]:
dx["pvd"] = dx["dx3_num"].between(440, 444).astype(int)

In [25]:
dx

,subject_id,hadm_id,seq_num,icd_code,icd_version,icd9_norm,ICD9,Description,HCUP CCS Classification,HCC Category,CCW Classification,CDPS Classification,dx3,hypertension_flag,dx3_num,heart_failure,stroke,pvd
0,10000032,22595853,1,5723,9,5723,5723,Portal hypertension,Oth liver dx,HCC25 End-Stage Liver Disease,NaN,"Gastrointestinal, Medium",572,0,572.0,0,0,0
1,10000032,22595853,2,78959,9,78959,78959,Ascites NEC,Oth liver dx,NaN,NaN,"Gastrointestinal, Medium",789,0,789.0,0,0,0
2,10000032,22595853,3,5715,9,5715,5715,Cirrhosis of liver NOS,Oth liver dx,HCC26 Cirrhosis of Liver,NaN,"Gastrointestinal, Medium",571,0,571.0,0,0,0
3,10000032,22595853,4,07070,9,07070,7070,Hpt C w/o hepat coma NOS,Hepatitis,NaN,NaN,"Infectious Disease, Low",070,0,70.0,0,0,0
4,10000032,22595853,5,496,9,496,496,Chr airway obstruct NEC,COPD,HCC108 Chronic Obstructive Pulmonary Disease,COPD,"Pulmonary, Low",496,0,496.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2870656,19999987,23865745,8,78039,9,78039,78039,Convulsions NEC,Epilepsy/cnv,NaN,NaN,Central Nervous System Low,780,0,780.0,0,0,0
2870657,19999987,23865745,9,0413,9,0413,413,Klebsiella pneumoniae,Oth bact inf,NaN,NaN,"Cardiovascular, Low",041,0,41.0,0,0,0
2870658,19999987,23865745,10,36846,9,36846,36846,Homonymous hemianopsia,Blindness,NaN,NaN,"Eye, Super Low",368,0,368.0,0,0,0
2870659,19999987,23865745,11,7810,9,7810,7810,Viral warts NOS,Viral infect,NaN,NaN,"Infectious Disease, Super Low",781,0,781.0,0,0,0


### Creating ICD for procedure
To merge the procedure information together, we started with **`procedures_icd.csv`**, which contains the procedures performed for each patient admission but only includes the procedure code and version (ICD-9 or ICD-10) without descriptions. To add meaningful information, we used **`icd_9_procedure_raw.csv`** as a lookup table for ICD-9 procedures and **`icd10 extra.xlsx`** as a lookup for ICD-10 procedures. First, we cleaned the column names and standardized the procedure codes by removing dots and leading zeros to ensure consistent matching across files. We then created common fields such as `icd_code`, `icd_version`, and a normalized code (`icd_norm`) in both lookup tables. After aligning the structure of the ICD-9 and ICD-10 lookup files, we combined them into a single procedure lookup table. Finally, we merged this combined lookup with the original **`procedures_icd.csv`** dataset using the procedure code and version so that each procedure record now includes its corresponding description and classification. The result is a single final dataset (`procedures_final`) that integrates all three files and provides a complete view of procedures with both codes and descriptive information.


In [23]:
procedures = pd.read_csv(r"C:\Users\Warner_Beast\OneDrive\Documents\CUNY\DATA 698\Data\Manual\procedures_icd.csv")
proc_raw_icd9 = pd.read_csv("icd_9_procedure_raw.csv")
icd10_extra = pd.read_excel("icd10 extra.xlsx")

In [24]:
procedures.columns = procedures.columns.str.strip()
proc_raw_icd9.columns = proc_raw_icd9.columns.str.strip()
icd10_extra.columns = icd10_extra.columns.str.strip()

In [25]:
# Read ICD-10 text file line by line
with open("icd10cm-codes-April-1-2026.txt", "r", encoding="utf-8") as f:
    icd10_lines = f.readlines()

# -----------------------------
# 2) Helper: normalize ICD code
# -----------------------------
def normalize_icd(x):
    x = str(x).strip().replace(".", "")
    x = x.lstrip("0")
    return x



In [26]:
print(proc_raw_icd9.columns.tolist())

['ICD9', 'Description', 'Classification']


In [27]:

proc_raw_icd9.columns = proc_raw_icd9.columns.str.strip()

In [28]:
# Clean column names
proc_raw_icd9.columns = proc_raw_icd9.columns.str.strip()

# Create standardized fields
proc_raw_icd9["icd_code"] = proc_raw_icd9["ICD9"].astype(str).str.strip()
proc_raw_icd9["icd_version"] = 9
proc_raw_icd9["icd_norm"] = proc_raw_icd9["icd_code"].apply(normalize_icd)

# Prepare lookup
proc_raw_icd9_lookup = proc_raw_icd9.rename(columns={
    "Description": "description",
    "Classification": "classification"
})[["icd_version", "icd_code", "icd_norm", "description", "classification"]]

# Check result
proc_raw_icd9_lookup.head()

,icd_version,icd_code,icd_norm,description,classification
0,9,1,1,Therapeutic ultrasound of vessels of head and ...,Ther ult head & neck ves
1,9,2,2,Therapeutic ultrasound of heart,Ther ultrasound of heart
2,9,3,3,Therapeutic ultrasound of peripheral vascular ...,Ther ult peripheral ves
3,9,9,9,Other therapeutic ultrasound,Other therapeutic ultsnd
4,9,10,10,Implantation of chemotherapeutic agent,Implant chemothera agent


In [29]:
# 5) Build ICD-10 lookup from icd10 extra.xlsx
# -----------------------------
icd10_extra["icd_code"] = icd10_extra["Code"].astype(str).str.strip()
icd10_extra["description"] = icd10_extra["Description"].astype(str).str.strip()
icd10_extra["icd_version"] = 10
icd10_extra["classification"] = pd.NA
icd10_extra["icd_norm"] = icd10_extra["icd_code"].apply(normalize_icd)

icd10_lookup = icd10_extra[[
    "icd_version", "icd_code", "icd_norm", "description", "classification"
]]


In [30]:
import re

# -----------------------------
# 4) Reformat ICD-10 text file
#    expected pattern:
#    CODE + spaces + DESCRIPTION
# -----------------------------
icd10_rows = []

for line in icd10_lines:
    line = line.strip()
    if not line:
        continue
    
    # Split on 2+ spaces or tab between code and description
    parts = re.split(r"\s{2,}|\t+", line, maxsplit=1)
    
    if len(parts) == 2:
        code, desc = parts
        code = code.strip()
        desc = desc.strip()
        
        # keep likely ICD-10 code rows only
        if re.match(r"^[A-TV-Z][0-9A-Z.]+$", code):
            icd10_rows.append({
                "icd_version": 10,
                "icd_code": code,
                "description": desc
            })

icd10_lookup = pd.DataFrame(icd10_rows)

In [31]:
procedure_lookup = pd.concat(
    [proc_raw_icd9_lookup, icd10_lookup],
    ignore_index=True
).drop_duplicates(subset=["icd_version", "icd_norm"])

# -----------------------------
# 7) Prepare procedures file
# -----------------------------
procedures["icd_code"] = procedures["icd_code"].astype(str).str.strip()
procedures["icd_norm"] = procedures["icd_code"].apply(normalize_icd)

# -----------------------------
# 8) Merge lookup to procedures
# -----------------------------
procedures_final = procedures.merge(
    procedure_lookup,
    on=["icd_version", "icd_norm"],
    how="left",
    suffixes=("", "_lookup")
)


In [32]:
procedures_final = procedures.merge(
    procedure_lookup,
    on=["icd_version", "icd_norm"],
    how="left",
    suffixes=("", "_lookup")
)

# -----------------------------
# 9) Save files
# -----------------------------
#procedure_lookup.to_csv("procedure_lookup_icd9_icd10_extra.csv", index=False)
#procedures_final.to_csv("procedures_joined_icd9_icd10_extra.csv", index=False)

# -----------------------------
# 10) Check results
# -----------------------------
print("ICD-9 lookup rows:", len(proc_raw_icd9_lookup))
print("ICD-10 extra rows:", len(icd10_lookup))
print("Combined lookup rows:", len(procedure_lookup))

print("\nMatch rate overall:")
print(procedures_final["description"].notna().mean())

print("\nSample:")
print(procedures_final.head())

ICD-9 lookup rows: 3882
ICD-10 extra rows: 23231
Combined lookup rows: 3810

Match rate overall:
0.6669277555968448

Sample:
   subject_id   hadm_id  seq_num   chartdate icd_code  icd_version icd_norm  \
0    10000032  22595853        1  2180-05-07     5491            9     5491   
1    10000032  22841357        1  2180-06-27     5491            9     5491   
2    10000032  25742920        1  2180-08-06     5491            9     5491   
3    10000068  25022803        1  2160-03-03     8938            9     8938   
4    10000117  27988844        1  2183-09-19  0QS734Z           10   QS734Z   

  icd_code_lookup                                  description  \
0            5491              Percutaneous abdominal drainage   
1            5491              Percutaneous abdominal drainage   
2            5491              Percutaneous abdominal drainage   
3            8938  Other nonoperative respiratory measurements   
4             NaN                                          NaN   

   

In [33]:
final_clean = procedures_final[
    ["subject_id", "hadm_id", "seq_num", "chartdate", "icd_code", "icd_version", "description", "classification"]
].copy()

In [34]:
final_clean

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version,description,classification
0,10000032,22595853,1,2180-05-07,5491,9,Percutaneous abdominal drainage,Percu abdominal drainage
1,10000032,22841357,1,2180-06-27,5491,9,Percutaneous abdominal drainage,Percu abdominal drainage
2,10000032,25742920,1,2180-08-06,5491,9,Percutaneous abdominal drainage,Percu abdominal drainage
3,10000068,25022803,1,2160-03-03,8938,9,Other nonoperative respiratory measurements,Respiratory measure NEC
4,10000117,27988844,1,2183-09-19,0QS734Z,10,NaN,NaN
...,...,...,...,...,...,...,...,...
668988,19999840,21033226,5,2164-09-16,0331,9,Spinal tap,Spinal tap
668989,19999840,26071774,1,2164-07-25,8891,9,Magnetic resonance imaging of brain and brain ...,Mri of brain & brainstem
668990,19999840,26071774,2,2164-07-25,8841,9,Arteriography of cerebral arteries,Contr cerebr arteriogram
668991,19999987,23865745,1,2145-11-07,8841,9,Arteriography of cerebral arteries,Contr cerebr arteriogram


In [35]:
icd10_lookup = pd.DataFrame(icd10_rows)
icd10_lookup["icd_norm"] = icd10_lookup["icd_code"].apply(normalize_icd)

# optional classification column so both lookups align
icd10_lookup["classification"] = pd.NA

# keep same columns as ICD-9 lookup
icd10_lookup = icd10_lookup[["icd_version", "icd_code", "icd_norm", "description", "classification"]]

# -----------------------------
# 5) Combine ICD-9 + ICD-10 lookups
# -----------------------------
procedure_lookup = pd.concat(
    [proc_raw_icd9_lookup, icd10_lookup],
    ignore_index=True
)

# remove duplicates if any
procedure_lookup = procedure_lookup.drop_duplicates(
    subset=["icd_version", "icd_norm"]
)

# -----------------------------
# 6) Prepare procedures file for join
# -----------------------------
procedures["icd_code"] = procedures["icd_code"].astype(str).str.strip()
procedures["icd_norm"] = procedures["icd_code"].apply(normalize_icd)

# -----------------------------
# 7) Join procedures to lookup
#    join on version + normalized code
# -----------------------------
procedures_final = procedures.merge(
    procedure_lookup,
    on=["icd_version", "icd_norm"],
    how="left",
    suffixes=("", "_lookup")
)

# -----------------------------
# 8) Save outputs
# -----------------------------
#procedure_lookup.to_csv("procedure_lookup_icd9_icd10.csv", index=False)
#procedures_final.to_csv("procedures_final_clean.csv", index=False)

# -----------------------------
# 9) Quick checks
# -----------------------------
match_rate = procedures_final["description"].notna().mean()
print(f"Match rate: {match_rate:.1%}")

print("\nFinal joined sample:")
print(procedures_final.head())

print("\nICD-10 reformatted sample:")
print(icd10_lookup.head())

Match rate: 66.7%

Final joined sample:
   subject_id   hadm_id  seq_num   chartdate icd_code  icd_version icd_norm  \
0    10000032  22595853        1  2180-05-07     5491            9     5491   
1    10000032  22841357        1  2180-06-27     5491            9     5491   
2    10000032  25742920        1  2180-08-06     5491            9     5491   
3    10000068  25022803        1  2160-03-03     8938            9     8938   
4    10000117  27988844        1  2183-09-19  0QS734Z           10   QS734Z   

  icd_code_lookup                                  description  \
0            5491              Percutaneous abdominal drainage   
1            5491              Percutaneous abdominal drainage   
2            5491              Percutaneous abdominal drainage   
3            8938  Other nonoperative respiratory measurements   
4             NaN                                          NaN   

             classification  
0  Percu abdominal drainage  
1  Percu abdominal drainage 

In [36]:
final_clean = procedures_final[[
    "subject_id",
    "hadm_id",
    "seq_num",
    "icd_code",
    "icd_version",
    "description",
    "classification"
]]

final_clean.to_csv("procedures_final_clean.csv", index=False)

,subject_id,hadm_id,seq_num,icd_code,icd_version,description,classification
0,10000032,22595853,1,5491,9,Percutaneous abdominal drainage,Percu abdominal drainage
1,10000032,22841357,1,5491,9,Percutaneous abdominal drainage,Percu abdominal drainage
2,10000032,25742920,1,5491,9,Percutaneous abdominal drainage,Percu abdominal drainage
3,10000068,25022803,1,8938,9,Other nonoperative respiratory measurements,Respiratory measure NEC
4,10000117,27988844,1,0QS734Z,10,NaN,NaN
...,...,...,...,...,...,...,...
668988,19999840,21033226,5,0331,9,Spinal tap,Spinal tap
668989,19999840,26071774,1,8891,9,Magnetic resonance imaging of brain and brain ...,Mri of brain & brainstem
668990,19999840,26071774,2,8841,9,Arteriography of cerebral arteries,Contr cerebr arteriogram
668991,19999987,23865745,1,8841,9,Arteriography of cerebral arteries,Contr cerebr arteriogram


### Service 

In [41]:
import pandas as pd
services = pd.read_csv('services.csv')
# Start from services table
svc = services.copy()

# Clean text
svc["curr_service"] = svc["curr_service"].astype(str).str.strip().str.upper()

# Example acuity ranking for services
# Update these names to match the values in your data
# Acuity map based on the service table you showed
acuity_map = {
    "TRAUM": 3,
    "CSURG": 3,
    "NSURG": 3,
    "TSURG": 3,
    "VSURG": 3,
    "SURG": 2,
    "ORTHO": 2,
    "PSURG": 2,
    "ENT": 2,
    "MED": 1,
    "CMED": 1,
    "OMED": 1,
    "NMED": 1,
    "GU": 1,
    "GYN": 1,
    "OBS": 1,
    "DENT": 0,
    "PSYCH": 0,
    "NB": 0,
    "NBB": 0
}

# Map service to acuity score
svc["acuity_score"] = svc["curr_service"].map(acuity_map).fillna(0)

# Optional binary flag: critical vs not critical
svc["acuity_flag"] = (svc["acuity_score"] >= 3).astype(int)

# Keep the most critical service per admission
svc_final = (
    svc.sort_values(["hadm_id", "acuity_score"], ascending=[True, False])
       .drop_duplicates(subset="hadm_id", keep="first")
       [["subject_id", "hadm_id", "curr_service", "acuity_score", "acuity_flag"]]
       .rename(columns={"curr_service": "most_critical_service"})
)

print(svc_final.head())

        subject_id   hadm_id most_critical_service  acuity_score  acuity_flag
21358     10467237  20000019                   MED           1.0            0
323866    16925328  20000024                   MED           1.0            0
441018    19430048  20000034                   MED           1.0            0
417353    18910522  20000041                 ORTHO           2.0            0
52866     11146739  20000057                   MED           1.0            0


In [42]:
svc_final = (
    svc.sort_values(["hadm_id", "acuity_score"], ascending=[True, False])
       .drop_duplicates("hadm_id")
       [["subject_id", "hadm_id", "curr_service", "acuity_score"]]
       .rename(columns={"curr_service": "most_critical_service"})
)

In [43]:
svc_final["high_acuity_flag"] = (svc_final["acuity_score"] >= 3).astype(int)

In [70]:
# print services clean data 
svc_final.to_csv("service_final_clean.csv", index=False)

### Merge All Dataset

In [57]:
# read CPT Events, microblap 
microbio = pd.read_csv(r'microbiologyevents.csv')
icu = pd.read_csv(r"ICUSTAYS.csv")
transfers = pd.read_csv(r'TRANSFERS.csv')
callout = pd.read_csv(r'CALLOUT.csv')
drcodes = pd.read_csv(r"DRGCODES.csv")
cptevents = pd.read_csv(r"CPTEVENTS.csv")
proc_events = pd.read_csv(r"PROCEDUREEVENTS_MV.csv")

C:\Users\Warner_Beast\AppData\Local\Temp\ipykernel_54292\4215954880.py:2: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  microbio = pd.read_csv(r'microbiologyevents.csv')
C:\Users\Warner_Beast\AppData\Local\Temp\ipykernel_54292\4215954880.py:7: DtypeWarning: Columns (4,5,7,11) have mixed types. Specify dtype option on import or set low_memory=False.
  cptevents = pd.read_csv(r"CPTEVENTS.csv")


In [58]:
microbio.columns = microbio.columns.str.lower()
proc_events.columns = proc_events.columns.str.lower()
icu.columns = icu.columns.str.lower()
transfers.columns = transfers.columns.str.lower()
callout.columns = callout.columns.str.lower()
drcodes.columns = drcodes.columns.str.lower()
cptevents.columns = cptevents.columns.str.lower()

In [73]:
chartevents = pd.read_csv(r'CHARTEVENTS.csv')
chartevents.columns = chartevents.columns.str.lower()


C:\Users\Warner_Beast\AppData\Local\Temp\ipykernel_54292\784633664.py:1: DtypeWarning: Columns (8,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  chartevents = pd.read_csv(r'CHARTEVENTS.csv')


In [74]:
chartevents.head()

,row_id,subject_id,hadm_id,icustay_id,itemid,charttime,storetime,cgid,value,valuenum,valueuom,warning,error,resultstatus,stopped
0,788,36,165660,241249.0,223834,2134-05-12 12:00:00,2134-05-12 13:56:00,17525.0,15.0,15.00,L/min,0.0,0.0,NaN,NaN
1,789,36,165660,241249.0,223835,2134-05-12 12:00:00,2134-05-12 13:56:00,17525.0,100.0,100.00,NaN,0.0,0.0,NaN,NaN
2,790,36,165660,241249.0,224328,2134-05-12 12:00:00,2134-05-12 12:18:00,20823.0,0.37,0.37,NaN,0.0,0.0,NaN,NaN
3,791,36,165660,241249.0,224329,2134-05-12 12:00:00,2134-05-12 12:19:00,20823.0,6.0,6.00,min,0.0,0.0,NaN,NaN
4,792,36,165660,241249.0,224330,2134-05-12 12:00:00,2134-05-12 12:19:00,20823.0,2.5,2.50,NaN,0.0,0.0,NaN,NaN


In [113]:
microbio.head(10)

,microevent_id,subject_id,hadm_id,micro_specimen_id,chartdate,charttime,spec_itemid,spec_type_desc,test_seq,storedate,...,org_name,isolate_num,quantity,ab_itemid,ab_name,dilution_text,dilution_comparison,dilution_value,interpretation,comments
0,1,10000032,NaN,636109,2180-03-23 00:00:00,2180-03-23 11:51:00,70093,Blood (Toxo),1,2180-03-26 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NEGATIVE FOR TOXOPLASMA IgG ANTIBODY BY EIA. ...
1,2,10000032,NaN,1836584,2180-03-23 00:00:00,2180-03-23 11:51:00,70017,SEROLOGY/BLOOD,1,2180-03-24 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSITIVE BY EIA. A positive IgG result genera...
2,3,10000032,NaN,4131591,2180-03-23 00:00:00,2180-03-23 11:51:00,70087,Blood (CMV AB),1,2180-03-26 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,___
3,4,10000032,NaN,4131591,2180-03-23 00:00:00,2180-03-23 11:51:00,70087,Blood (CMV AB),2,2180-03-26 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NEGATIVE FOR CMV IgM ANTIBODY BY EIA. INTERPR...
4,5,10000032,NaN,6028147,2180-03-23 00:00:00,2180-03-23 11:51:00,70088,Blood (EBV),1,2180-03-25 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSITIVE BY EIA.
5,6,10000032,NaN,6028147,2180-03-23 00:00:00,2180-03-23 11:51:00,70088,Blood (EBV),2,2180-03-25 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSITIVE BY EIA.
6,7,10000032,NaN,6028147,2180-03-23 00:00:00,2180-03-23 11:51:00,70088,Blood (EBV),3,2180-03-25 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NEGATIVE <1:10 BY IFA. INTERPRETATION: RESULT...
7,8,10000032,NaN,6700137,2180-03-23 00:00:00,2180-03-23 11:51:00,70017,SEROLOGY/BLOOD,1,2180-03-24 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSITIVE BY EIA. A positive IgG result genera...
8,9,10000032,NaN,7713497,2180-03-23 00:00:00,2180-03-23 11:51:00,70046,IMMUNOLOGY,1,2180-03-26 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HIV-1 RNA is not detected. Performed using th...
9,10,10000032,NaN,7951493,2180-03-23 00:00:00,2180-03-23 11:51:00,70017,SEROLOGY/BLOOD,1,2180-03-26 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSITIVE BY EIA. A positive IgG result genera...


In [125]:
admissions.head(10)

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,hypertension_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,ENGLISH,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,ENGLISH,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,ENGLISH,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,EMERGENCY ROOM,NaN,Other,ENGLISH,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0,0
5,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,NaN,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,ENGLISH,MARRIED,WHITE,2160-11-20 20:36:00,2160-11-21 03:20:00,0,0
6,10000084,29888819,2160-12-28 05:11:00,2160-12-28 16:07:00,NaN,EU OBSERVATION,PHYSICIAN REFERRAL,NaN,Medicare,ENGLISH,MARRIED,WHITE,2160-12-27 18:32:00,2160-12-28 16:07:00,0,0
7,10000108,27250926,2163-09-27 23:17:00,2163-09-28 09:04:00,NaN,EU OBSERVATION,EMERGENCY ROOM,NaN,Other,ENGLISH,SINGLE,WHITE,2163-09-27 16:18:00,2163-09-28 09:04:00,0,0
8,10000117,22927623,2181-11-15 02:05:00,2181-11-15 14:52:00,NaN,EU OBSERVATION,EMERGENCY ROOM,NaN,Other,ENGLISH,DIVORCED,WHITE,2181-11-14 21:51:00,2181-11-15 09:57:00,0,0
9,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,NaN,OBSERVATION ADMIT,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Other,ENGLISH,DIVORCED,WHITE,2183-09-18 08:41:00,2183-09-18 20:20:00,0,0


In [59]:
proc_events.head()

,row_id,subject_id,hadm_id,icustay_id,starttime,endtime,itemid,value,valueuom,location,...,ordercategoryname,secondaryordercategoryname,ordercategorydescription,isopenbag,continueinnextdept,cancelreason,statusdescription,comments_editedby,comments_canceledby,comments_date
0,379,29070,115071,232563.0,2145-03-12 23:04:00,2145-03-12 23:05:00,225401,1.0,None,NaN,...,Procedures,NaN,Electrolytes,0,0,0,FinishedRunning,NaN,NaN,NaN
1,380,29070,115071,232563.0,2145-03-12 23:04:00,2145-03-12 23:05:00,225454,1.0,None,NaN,...,Procedures,NaN,Electrolytes,0,0,0,FinishedRunning,NaN,NaN,NaN
2,381,29070,115071,232563.0,2145-03-12 23:05:00,2145-03-18 20:01:00,225792,8456.0,hour,NaN,...,Ventilation,NaN,Task,1,0,0,FinishedRunning,NaN,NaN,NaN
3,382,29070,115071,232563.0,2145-03-12 23:36:00,2145-03-12 23:37:00,225402,1.0,None,NaN,...,Procedures,NaN,Electrolytes,0,0,0,FinishedRunning,NaN,NaN,NaN
4,383,29070,115071,232563.0,2145-03-13 01:27:00,2145-03-16 16:00:00,224560,5193.0,min,Right IJ,...,Invasive Lines,NaN,Task,1,0,0,FinishedRunning,NaN,NaN,NaN


In [47]:
diagnosis = pd.read_csv('diagnosis_main.csv')
# Keep only one diagnosis row per admission

diagnosis.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,icd9_norm,ICD9,Description,HCUP CCS Classification,HCC Category,CCW Classification,CDPS Classification,dx3
0,10000032,22595853,1,5723,9,5723,5723,Portal hypertension,Oth liver dx,HCC25 End-Stage Liver Disease,NaN,"Gastrointestinal, Medium",572
1,10000032,22595853,2,78959,9,78959,78959,Ascites NEC,Oth liver dx,NaN,NaN,"Gastrointestinal, Medium",789
2,10000032,22595853,3,5715,9,5715,5715,Cirrhosis of liver NOS,Oth liver dx,HCC26 Cirrhosis of Liver,NaN,"Gastrointestinal, Medium",571
3,10000032,22595853,4,07070,9,7070,7070,Hpt C w/o hepat coma NOS,Hepatitis,NaN,NaN,"Infectious Disease, Low",707
4,10000032,22595853,5,496,9,496,496,Chr airway obstruct NEC,COPD,HCC108 Chronic Obstructive Pulmonary Disease,COPD,"Pulmonary, Low",496


In [164]:
procedures_final.head()

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version,icd_norm,icd_code_lookup,description,classification
0,10000032,22595853,1,2180-05-07,5491,9,5491,5491,Percutaneous abdominal drainage,Percu abdominal drainage
1,10000032,22841357,1,2180-06-27,5491,9,5491,5491,Percutaneous abdominal drainage,Percu abdominal drainage
2,10000032,25742920,1,2180-08-06,5491,9,5491,5491,Percutaneous abdominal drainage,Percu abdominal drainage
3,10000068,25022803,1,2160-03-03,8938,9,8938,8938,Other nonoperative respiratory measurements,Respiratory measure NEC
4,10000117,27988844,1,2183-09-19,0QS734Z,10,QS734Z,NaN,NaN,NaN


In [52]:
# 1) Standardize column names
# -----------------------------
for df in [admissions, diagnosis, procedures_final, microbio]:
    df.columns = df.columns.str.lower()

# -----------------------------
# 2) Standardize key columns
# -----------------------------
for df in [admissions, diagnosis, procedures_final, microbio,icu,transfers,callout,drcodes,cptevents]:
    df["subject_id"] = pd.to_numeric(df["subject_id"], errors="coerce").astype("Int64")
    df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce").astype("Int64")

In [50]:
# -----------------------------
proc_summary = (
    procedures_final.groupby(["subject_id", "hadm_id"], as_index=False)
    .agg(
        procedure_count=("seq_num", "count"),
        unique_procedure_seq=("seq_num", "nunique")
    )
)

# 5) Microbiology summary at admission level
# -----------------------------
micro_summary = (
    microbio.groupby(["subject_id", "hadm_id"], as_index=False)
    .agg(
        micro_event_count=("microevent_id", "count"),
        unique_micro_specimens=("micro_specimen_id", "nunique") if "micro_specimen_id" in microbio.columns else ("microevent_id", "count")
    )
)

diag_summary = (
    diagnosis.groupby(["subject_id", "hadm_id"], as_index=False)
    .agg(
        diagnosis_count=("icd_code", "count"),
        unique_diagnosis_codes=("icd_code", "nunique"),
        first_diag_code=("icd_code", "first")
    )
)


In [60]:
proc_events_agg = (
    proc_events.groupby(["subject_id", "hadm_id","ordercategoryname"])
    .agg(
        proc_event_count=("subject_id", "size"),
        proc_unique_items=("itemid", "nunique")
    )
    .reset_index()
)

In [75]:
chartevents_agg = (
    chartevents
    .groupby(["subject_id", "hadm_id"])
    .agg(
        chart_event_count=("itemid", "count"),
        chart_unique_items=("itemid", "nunique"),
        chart_avg_value=("valuenum", "mean"),
        chart_max_value=("valuenum", "max"),
        chart_min_value=("valuenum", "min")
    )
    .reset_index()
)

In [78]:
chartevents_agg = chartevents_agg.fillna(0)

In [79]:
vital_itemids = [
    211, 220045,       # heart rate
    51, 442, 455, 6701, 220179,   # systolic BP
    8368, 8441, 220180,           # diastolic BP
    456, 52, 6702, 443, 220181,   # mean BP
    618, 615, 220210,             # respiratory rate
    223761, 678,                  # temperature
    646, 220277,                  # oxygen saturation
    807, 811, 1529, 3745          # glucose
]

vitals = chartevents[chartevents["itemid"].isin(vital_itemids)]

In [80]:
chartevents_agg = (
    vitals
    .groupby(["subject_id", "hadm_id"])
    .agg(
        vitals_count=("itemid", "count"),
        vitals_unique=("itemid", "nunique"),
        vitals_mean=("valuenum", "mean"),
        vitals_max=("valuenum", "max"),
        vitals_min=("valuenum", "min")
    )
    .reset_index()
)

In [81]:
chartevents_agg

,subject_id,hadm_id,vitals_count,vitals_unique,vitals_mean,vitals_max,vitals_min
0,2,163353,5,2,127.000000,148.0,72.0
1,3,145834,747,9,74.073647,306.0,0.0
2,4,185777,125,8,93.034132,266.0,18.0
3,5,178980,2,2,116.500000,140.0,93.0
4,6,107064,416,9,76.062330,254.0,8.0
...,...,...,...,...,...,...,...
22931,99985,176670,1027,7,76.065823,168.0,14.0
22932,99991,151118,529,7,89.022495,194.0,11.0
22933,99992,197084,276,7,75.751449,172.0,11.0
22934,99995,137810,204,7,67.450000,136.0,11.0


In [62]:
transfers.head()

,row_id,subject_id,hadm_id,icustay_id,dbsource,eventtype,prev_careunit,curr_careunit,prev_wardid,curr_wardid,intime,outtime,los
0,657,111,192123,254245.0,carevue,transfer,CCU,MICU,7.0,23.0,2142-04-29 15:27:11,2142-05-04 20:38:33,125.19
1,658,111,192123,NaN,carevue,transfer,MICU,NaN,23.0,45.0,2142-05-04 20:38:33,2142-05-05 11:46:32,15.13
2,659,111,192123,NaN,carevue,discharge,NaN,NaN,45.0,NaN,2142-05-05 11:46:32,NaN,NaN
3,660,111,155897,249202.0,metavision,admit,NaN,MICU,NaN,52.0,2144-07-01 04:13:59,2144-07-01 05:19:39,1.09
4,661,111,155897,NaN,metavision,transfer,MICU,NaN,52.0,32.0,2144-07-01 05:19:39,2144-07-01 06:28:29,1.15


In [70]:
# 4) Aggregate ICU
icu_agg = (
    icu.groupby(["subject_id", "hadm_id"])
    .agg(
        icu_stay_count=("icustay_id", "nunique")
    )
    .reset_index()
)

icu_agg["icu_flag"] = 1

# 6) Aggregate callout
callout_agg = (
    callout.groupby(["subject_id", "hadm_id"])
    .agg(
        callout_count=("subject_id", "size")
    )
    .reset_index()
)

callout_agg["callout_flag"] = 1 

# 7) Aggregate DRG codes
drcodes_agg = (
    drcodes.groupby(["subject_id", "hadm_id"])
    .agg(
        drg_count=("subject_id", "size"),
        drg_unique_codes=("drg_code", "nunique")
    )
    .reset_index()
)

In [71]:
# 8) Aggregate CPT events
cptevents_agg = (
    cptevents.groupby(["subject_id", "hadm_id"])
    .agg(
        cpt_count=("subject_id", "size"),
        cpt_unique_codes=("cpt_cd", "nunique")
    )
    .reset_index()
)

In [86]:


# -----------------------------
# 5) Summarize DRGCODES at hadm_id level
# -----------------------------
drg_summary = (
    drcodes.groupby(["subject_id", "hadm_id"], as_index=False)
    .agg(
        drg_count=("drg_code", "count"),
        
        
        drg_code_first=("drg_code", "first"),
        drg_desc_first=("description", "first"),
        drg_severity_max=("drg_severity", "max"),
       
        drg_mortality_max=("drg_mortality", "max"),
        
    )
)

In [124]:
svc_agg = (
    svc_final.groupby(["subject_id", "hadm_id"], as_index=False)
    .agg(
        service_count=("most_critical_service", "count"),
        unique_services=("most_critical_service", "nunique"),
        most_critical_service=("most_critical_service", "first"),
        acuity_score=("acuity_score", "max"),
        high_acuity_flag=("high_acuity_flag", "max")
    )
)

In [96]:
# 5) Aggregate transfers
transfers_agg = (
    transfers.groupby(["subject_id", "hadm_id"])
    .agg(
        transfer_count=("subject_id", "size"),
        transfer_unique_careunits=("curr_careunit", "nunique")
    )
    .reset_index()
)

In [91]:
num_cols = drg_summary.select_dtypes(include="number").columns
drg_summary[num_cols] = drg_summary[num_cols].fillna(0)

In [92]:
drg_summary

,subject_id,hadm_id,drg_count,drg_code_first,drg_desc_first,drg_severity_max,drg_mortality_max
0,2,163353,2,6402,"Neonate, Bwt > 2499g, Normal Newborn Or Neonat...",2.0,1.0
1,3,145834,1,416,SEPTICEMIA AGE >17,0.0,0.0
2,4,185777,1,489,HIV WITH MAJOR RELATED CONDITION,0.0,0.0
3,5,178980,1,391,NORMAL NEWBORN,0.0,0.0
4,6,107064,1,302,KIDNEY TRANSPLANT,0.0,0.0
...,...,...,...,...,...,...,...
58885,99985,176670,3,7204,Septicemia & Disseminated Infections,4.0,4.0
58886,99991,151118,3,2214,Major Small & Large Bowel Procedures,4.0,4.0
58887,99992,197084,3,907,OTHER O.R. PROCEDURES FOR INJURIES W MCC,4.0,4.0
58888,99995,137810,3,1732,Other Vascular Procedures,2.0,3.0


In [94]:
for name, table in {
    "microbio": microbio,
    "proc_events": proc_events,
    "icu": icu,
    "transfers": transfers,
    "callout": callout,
    "drcodes": drcodes,
    "cptevents": cptevents
}.items():
    dupes = table.duplicated(["subject_id", "hadm_id"]).sum()
    print(name, "duplicate admission rows:", dupes)

microbio duplicate admission rows: 2900226
proc_events duplicate admission rows: 236172
icu duplicate admission rows: 3746
transfers duplicate admission rows: 202921
callout duplicate admission rows: 5767
drcodes duplicate admission rows: 66667
cptevents duplicate admission rows: 528998


In [95]:
# 1) Start from admissions-level table
base = admissions[["subject_id", "hadm_id"]].drop_duplicates().copy()

In [101]:
base.head()

,subject_id,hadm_id
0,10000032,22595853
1,10000032,22841357
2,10000032,25742920
3,10000032,29079034
4,10000068,25022803


In [120]:
df = base.merge(micro_summary, on=["subject_id", "hadm_id"], how="left")


In [ ]:
df

,subject_id,hadm_id,micro_event_count,unique_micro_specimens
0,10000032,22595853,5.0,3.0
1,10000032,22841357,2.0,2.0
2,10000032,25742920,1.0,1.0
3,10000032,29079034,2.0,2.0
4,10000068,25022803,NaN,NaN


In [125]:

df = df.merge(micro_summary, on=["subject_id", "hadm_id"], how="left")
df = df.merge(proc_summary, on=["subject_id", "hadm_id"], how="left")
df = df.merge(svc_agg, on=["subject_id", "hadm_id"], how="left")




In [119]:
proc_events_agg

,subject_id,hadm_id,ordercategoryname,proc_event_count,proc_unique_items
0,23,124321,Invasive Lines,2,2
1,23,124321,Peripheral Lines,2,1
2,34,144319,Imaging,1,1
3,34,144319,Intubation/Extubation,1,1
4,34,144319,Invasive Lines,1,1
...,...,...,...,...,...
90569,99995,137810,Peripheral Lines,1,1
90570,99995,137810,Procedures,2,2
90571,99995,137810,Significant Events,1,1
90572,99995,137810,Ventilation,1,1


In [ ]:
df = df.merge(icu_agg, on=["subject_id", "hadm_id"], how="left")
df = df.merge(transfers_agg, on=["subject_id", "hadm_id"], how="left")
df = df.merge(callout_agg, on=["subject_id", "hadm_id"], how="left")
df = df.merge(drcodes_agg, on=["subject_id", "hadm_id"], how="left")
df = df.merge(cptevents_agg, on=["subject_id", "hadm_id"], how="left")
df = df.merge(proc_events_agg, on=["subject_id", "hadm_id"], how="left")
df = df.merge(micro_summary, on=["subject_id", "hadm_id"], how="left")

In [126]:
df.isna().sum()

subject_id                       0
hadm_id                          0
micro_event_count_x         267245
unique_micro_specimens_x    267245
micro_event_count_y         267245
unique_micro_specimens_y    267245
procedure_count_x           201832
unique_procedure_seq_x      201832
micro_event_count           267245
unique_micro_specimens      267245
procedure_count_y           201832
unique_procedure_seq_y      201832
service_count                    0
unique_services                  0
most_critical_service            0
acuity_score                     0
high_acuity_flag                 0
dtype: int64

In [106]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 431088 entries, 0 to 431087
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   subject_id                 431088 non-null  Int64  
 1   hadm_id                    431088 non-null  Int64  
 2   micro_event_count          163843 non-null  float64
 3   unique_micro_specimens     163843 non-null  float64
 4   ordercategoryname          0 non-null       object 
 5   proc_event_count           0 non-null       float64
 6   proc_unique_items          0 non-null       float64
 7   icu_stay_count             0 non-null       float64
 8   icu_flag                   0 non-null       float64
 9   transfer_count             0 non-null       float64
 10  transfer_unique_careunits  0 non-null       float64
 11  callout_count              0 non-null       float64
 12  callout_flag               0 non-null       float64
 13  drg_count                  0 